# VertiRTA-X8-240 — 02. ToS-Clean Istanbul 4K30 Blender Cycles OptiX Cinema

**Recommended GPU:** **Google Colab `L4 (24 GB)`** (Ada Lovelace `sm_89` RT cores + hardware `hevc_nvenc`).
**Do NOT allocate `G4 (96 GB)`:** The 3-frame Cycles OptiX preflight strictly enforces **`< 18.0 GB` VRAM**, leaving `>= 6 GB` headroom on `L4 (24 GB)`. Avoid `T4 (16 GB)` because a full 4K scene can exceed 16 GB.

### Pipeline Guarantees (Aligned with `viz/blender/` & `viz/geo/`)
- **Driver-Gated Blender LTS:** Automatically selects **Blender 5.2 LTS** (`driver >= 575`) or **Blender 4.5 LTS** (`driver 535..574`), rejecting 4.2/4.3.
- **ToS-Clean Corridor Data:** Raw AWS Copernicus DEM GLO-30 (`N40_00_E029_00`, `-dstnodata -9999`) + raw AWS `sentinel-cogs` L2A (`35TPF`) warped into Kartal local `tmerc` (NEVER `EPSG:32635`, EOX, FABDEM, Bing, Esri, or Google).
- **Resumable 150-Frame Slices:** Syncs completed frames to Drive every 150 frames (`12` slices for `1,800` frames = `60 s @ 30 fps`) and muxes synced RGBA telemetry HUD via **`hevc_nvenc -preset p5`** (legacy `-preset slow` hard-banned).

In [ ]:
import subprocess

out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip()
name, driver_ver, mem_mb_str = [part.strip() for part in out.splitlines()[0].split(",")]
vram_gb = float(mem_mb_str) / 1024.0
print(f"Detected GPU: {name} | Driver: {driver_ver} | VRAM: {vram_gb:.1f} GB")

if vram_gb >= 60.0:
    print(
        "[COST WARNING] You attached a >=60 GB GPU (e.g., G4 96 GB). "
        "This 4K OptiX render is preflight-gated to < 18 GB VRAM and runs cleanly on L4 (24 GB). "
        "Switch to L4 (24 GB) to save Colab Compute Units!"
    )
elif vram_gb < 20.0:
    print(
        f"[VRAM WARNING] Detected {vram_gb:.1f} GB VRAM (< 20 GB). "
        "L4 (24 GB) is strongly recommended for the < 18 GB 4K Cycles OptiX preflight budget."
    )
else:
    print("GPU tier right-sized for VertiRTA 4K Cycles OptiX + hevc_nvenc p5 (L4 24 GB).")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Google Colab; skipping drive.mount().")

REPO_URL = os.environ.get("VERTIRTA_REPO_URL", "https://github.com/vertirta/vertirta.git")
REPO_DIR = Path("/content/vertirta")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow", "pyyaml", "numpy"], check=True)
print("Workspace ready:", Path.cwd())

In [ ]:
import subprocess
from viz.blender.blender_env_check import (
    build_colab_install_commands,
    probe_host_nvidia_driver_major,
    select_blender_release,
)

driver_major = probe_host_nvidia_driver_major()
release = select_blender_release(driver_major)
print(
    f"Detected NVIDIA driver {driver_major} -> Blender {release['version']} LTS "
    f"(min_driver={release['min_driver']})"
)
for cmd in build_colab_install_commands(driver_major):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
from pathlib import Path
from viz.geo.airspace_polygons import export_airspace_geojson
from viz.geo.build_istanbul_corridor import build_corridor_pipeline_plan

geo_dir = Path("/content/work/geo_data")
geo_dir.mkdir(parents=True, exist_ok=True)
plan = build_corridor_pipeline_plan(str(geo_dir))
export_airspace_geojson(str(geo_dir / "istanbul_ltfj_ltbx_ctr.geojson"))
print("Verified Kartal local tmerc CRS:", plan["local_tmerc_crs"])
print("GLO-30 tile:", plan["tiles"]["copernicus_glo30"], "| Sentinel-2 tile:", plan["tiles"]["sentinel2_mgrs"])

In [ ]:
import os
import subprocess
from pathlib import Path

Path("/content/work").mkdir(parents=True, exist_ok=True)
scene_expr = (
    "import bpy; "
    "from viz.blender.build_istanbul_scene import "
    "build_turkish_rooftop_concrete_slz_shader, build_terracotta_roof_shader, "
    "build_marmara_ocean_shader, build_nishita_istanbul_sky, build_ctr_curtain_object; "
    "build_turkish_rooftop_concrete_slz_shader(bpy); "
    "build_terracotta_roof_shader(bpy); "
    "build_marmara_ocean_shader(bpy); "
    "build_nishita_istanbul_sky(bpy); "
    "build_ctr_curtain_object(bpy, 'ltfj'); "
    "build_ctr_curtain_object(bpy, 'ltbx'); "
    "bpy.ops.wm.save_as_mainfile(filepath='/content/work/vertirta_istanbul_master.blend')"
)
env = dict(os.environ)
env["PYTHONPATH"] = str(Path.cwd())
subprocess.run(["blender", "-b", "--python-expr", scene_expr], check=True, env=env)
print("Saved master scene: /content/work/vertirta_istanbul_master.blend")

In [ ]:
import os
import subprocess
from pathlib import Path
from viz.blender.render_slices import (
    assert_vram_within_budget,
    build_preflight_3frame_cmd,
    parse_cycles_peak_vram_gb,
)

VRTA_LOG = Path(
    os.environ.get("VERTIRTA_VRTA_LOG", "/content/drive/MyDrive/VertiRTA_Artifacts/logs/kartal_m3_fault.vrta")
)
assert VRTA_LOG.exists(), f"Upload a 60 s .vrta log first (expected at {VRTA_LOG})."

pf_cmd = build_preflight_3frame_cmd(
    "/content/work/vertirta_istanbul_master.blend",
    "viz/blender/apply_vrta_and_render_setup.py",
    "/content/preflight/pf_####",
    str(VRTA_LOG),
)
assert "OPTIX" in pf_cmd, "Cycles OPTIX device flag missing"
proc = subprocess.run(pf_cmd, capture_output=True, text=True, check=True)
peak_gb = parse_cycles_peak_vram_gb(proc.stdout + "\n" + proc.stderr)
assert_vram_within_budget(peak_gb, max_gb=18.0)
print(f"Preflight PASS: Peak VRAM = {peak_gb:.2f} GB (< 18.0 GB L4 budget)")

In [ ]:
import os
import subprocess
from pathlib import Path
from viz.blender.render_slices import (
    build_blender_slice_cmd,
    build_slice_rsync_cmd,
    compute_frame_slices,
)

VRTA_LOG = Path(
    os.environ.get("VERTIRTA_VRTA_LOG", "/content/drive/MyDrive/VertiRTA_Artifacts/logs/kartal_m3_fault.vrta")
)
out_dir = Path("/content/renders_4k")
drive_dir = Path("/content/drive/MyDrive/VertiRTA_Artifacts/renders_4k_frames")
out_dir.mkdir(parents=True, exist_ok=True)
drive_dir.mkdir(parents=True, exist_ok=True)

# Restore any previously rendered frames from Drive so interrupted sessions resume cleanly
subprocess.run(build_slice_rsync_cmd(drive_dir, out_dir), check=True)
completed = {int(p.stem.split("_")[-1]) for p in out_dir.glob("frame_*.png")}
remaining_slices = compute_frame_slices(total_frames=1800, slice_size=150, completed_frames=completed)
print(f"Resumable render: {len(completed)}/1800 frames done, {len(remaining_slices)} slices remaining.")

for start_f, end_f in remaining_slices:
    cmd = build_blender_slice_cmd(
        "/content/work/vertirta_istanbul_master.blend",
        "viz/blender/apply_vrta_and_render_setup.py",
        str(out_dir / "frame_####"),
        str(VRTA_LOG),
        start_f,
        end_f,
    )
    subprocess.run(cmd, check=True)
    subprocess.run(build_slice_rsync_cmd(out_dir, drive_dir), check=True)
    print(f"Synced slice [{start_f}, {end_f}] to {drive_dir}")

In [ ]:
import hashlib
import os
import shutil
import subprocess
from pathlib import Path

from viz.blender.mux_hud_ffmpeg import (
    build_ffmpeg_nvenc_cmd,
    format_hud_telemetry_payload,
    render_hud_frame_rgba,
)
from viz.cesium.vrta_reader import read_vrta_file

VRTA_LOG = Path(
    os.environ.get("VERTIRTA_VRTA_LOG", "/content/drive/MyDrive/VertiRTA_Artifacts/logs/kartal_m3_fault.vrta")
)
hud_dir = Path("/content/hud_4k")
hud_dir.mkdir(parents=True, exist_ok=True)
out_mp4 = Path("/content/out/VertiRTA_X8_240_Istanbul_4K30_FlightReplay.mp4")
out_mp4.parent.mkdir(parents=True, exist_ok=True)

_, frames_100hz = read_vrta_file(VRTA_LOG, verify_crc=True)
total_video_frames = 1800
for video_idx in range(1, total_video_frames + 1):
    src_idx = min(len(frames_100hz) - 1, int(round((video_idx - 1) * (100.0 / 30.0))))
    payload = format_hud_telemetry_payload(frames_100hz[src_idx], video_idx, total_video_frames)
    render_hud_frame_rgba(payload, hud_dir / f"hud_{video_idx:04d}.png", width=3840, height=2160)

ff_cmd = build_ffmpeg_nvenc_cmd(
    "/content/renders_4k/frame_%04d.png",
    "/content/hud_4k/hud_%04d.png",
    str(out_mp4),
    fps=30,
    preset="p5",
)
print("Muxing 4K30 cinema:", " ".join(ff_cmd))
subprocess.run(ff_cmd, check=True)

drive_video = Path("/content/drive/MyDrive/VertiRTA_Artifacts/cinema") / out_mp4.name
drive_video.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_mp4, drive_video)
digest = hashlib.sha256(out_mp4.read_bytes()).hexdigest()
print(f"Saved {drive_video} ({out_mp4.stat().st_size:,} B) | SHA-256: {digest}")